In [ ]:
# M1: FAILURE MODE CLASSIFIER
# XGBoost multi-class: predicts bearing_wear, thermal_degradation, imbalance, misalignment, normal

import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import json

session = get_active_session()
session.sql("USE DATABASE FAILURE_GENOME_DB").collect()
session.sql("USE SCHEMA ML_MODELS").collect()
print("Session ready.")

In [ ]:
# Load from the training view we created in Phase 2
df = session.table("FAILURE_GENOME_DB.ML_FEATURES.TRAIN_FAILURE_MODE").to_pandas()
print(f"Loaded {len(df)} rows")
print(f"Class distribution:\n{df['FAILURE_MODE'].value_counts()}")

In [ ]:
# Feature columns (exclude identifiers and targets)
exclude_cols = ['ASSET_ID', 'TIMESTAMP', 'FAILURE_MODE', 'DEGRADATION_STAGE']
feature_cols = [c for c in df.columns if c not in exclude_cols]
print(f"Using {len(feature_cols)} features")

# Handle nulls and infinities
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

# Encode target
le = LabelEncoder()
df['TARGET'] = le.fit_transform(df['FAILURE_MODE'])
print(f"Classes: {list(le.classes_)}")

# Time-based split (80/20)
df = df.sort_values('TIMESTAMP')
split_idx = int(len(df) * 0.8)
X_train, X_test = df.iloc[:split_idx][feature_cols].values, df.iloc[split_idx:][feature_cols].values
y_train, y_test = df.iloc[:split_idx]['TARGET'].values, df.iloc[split_idx:]['TARGET'].values
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
# Train XGBoost multi-class classifier
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softprob',
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print("Training complete.")

In [ ]:
# Predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

# Metrics
f1 = f1_score(y_test, y_pred, average='macro')
print(f"F1 Macro: {f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(pd.DataFrame(cm, index=le.classes_, columns=le.classes_))

In [ ]:
# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

print("Top 15 Features:")
print(importance.to_string(index=False))

In [ ]:
import os
from snowflake.ml.registry import Registry
from snowflake.ml.model import custom_model

# Save XGBoost model to file (fixes closure bug)
model_path = "/tmp/failure_mode_xgb.json"
model.save_model(model_path)

# Save label encoder classes
classes_list = list(le.classes_)
print(f"Model saved. Classes: {classes_list}")

class FailureModeModel(custom_model.CustomModel):
    def __init__(self, context):
        super().__init__(context)

    @custom_model.inference_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        import numpy as np
        import xgboost as xgb

        # Load model from artifact (NOT outer scope)
        xgb_model = xgb.XGBClassifier()
        xgb_model.load_model(self.context.path("xgb_model"))

        exclude = {'ASSET_ID', 'TIMESTAMP', 'FAILURE_MODE', 'DEGRADATION_STAGE'}
        cols = [c for c in input_df.columns if c not in exclude]
        X = input_df[cols].replace([np.inf, -np.inf], np.nan).fillna(0).values

        preds = xgb_model.predict(X)
        proba = xgb_model.predict_proba(X)

        classes = ['bearing_wear', 'imbalance', 'misalignment', 'normal', 'thermal_degradation']

        return pd.DataFrame({
            'PREDICTED_FAILURE_MODE': [classes[p] for p in preds],
            'CONFIDENCE': proba.max(axis=1).round(4),
            'PROB_BEARING_WEAR': proba[:, 0].round(4),
            'PROB_IMBALANCE': proba[:, 1].round(4),
            'PROB_MISALIGNMENT': proba[:, 2].round(4),
            'PROB_NORMAL': proba[:, 3].round(4),
            'PROB_THERMAL_DEGRADATION': proba[:, 4].round(4)
        })

# Register with artifact
reg = Registry(session=session, database_name="FAILURE_GENOME_DB", schema_name="ML_MODELS")

try:
    reg.delete_model("FAILURE_MODE_CLASSIFIER")
    print("Deleted existing model.")
except:
    pass

fm_model = FailureModeModel(
    context=custom_model.ModelContext(
        artifacts={"xgb_model": model_path}
    )
)
sample_input = pd.DataFrame(np.zeros((1, len(feature_cols))), columns=feature_cols)

mv = reg.log_model(
    fm_model,
    model_name="FAILURE_MODE_CLASSIFIER",
    version_name="V1",
    metrics={"f1_macro": float(f1)},
    sample_input_data=sample_input,
    target_platforms=["WAREHOUSE"],
    comment=f"XGBoost 5-class failure mode classifier. F1={f1:.4f}"
)
print(f"Model registered: FAILURE_MODE_CLASSIFIER V1 | F1: {f1:.4f}")

In [ ]:
# Test inference from registry
reg_model = reg.get_model("FAILURE_MODE_CLASSIFIER").version("V1")
test_sample = pd.DataFrame(X_test[:5], columns=feature_cols)
result = reg_model.run(test_sample, function_name="predict")
print("Inference test:")
print(result)
print(f"\nActual: {le.inverse_transform(y_test[:5])}")